# EpigraNet Tamil OCR with Streamlit on Google Colab

This notebook launches a Streamlit frontend for the EpigraNet OCR pipeline inside Google Colab.
The app can download its model checkpoint, reference embeddings, and class mapping from Hugging Face Hub.

Before you run it:

1. Push this project to GitHub.
2. Upload these runtime assets to a Hugging Face model repository:
   - `epigranet_embedding_model (1).pt`
   - `reference_embeddings.pt`
   - `class_mapping_209 (1).json`
3. Update `REPO_URL` and `HF_REPO_ID` in the config cell.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<your-github-user>/epirgranet_implementation_full.git"
PROJECT_DIR = Path("/content/epirgranet_implementation_full")

HF_REPO_ID = "your-hf-username/epigranet-tamil-ocr"
HF_REVISION = "main"
HF_TOKEN = None  # Set a token string here if the Hugging Face repo is private.

MODEL_FILENAME = "epigranet_embedding_model (1).pt"
EMBEDDINGS_FILENAME = "reference_embeddings.pt"
CLASS_MAPPING_FILENAME = "class_mapping_209 (1).json"

STREAMLIT_PORT = 8501


In [ ]:
import subprocess
import sys

if PROJECT_DIR.exists():
    print(f"Using existing project directory: {PROJECT_DIR}")
else:
    if "<your-github-user>" in REPO_URL:
        raise ValueError("Update REPO_URL in the config cell before running setup.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

required_files = [
    PROJECT_DIR / "pipeline.py",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "streamlit_app.py",
]
missing_files = [str(path.relative_to(PROJECT_DIR)) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "The cloned GitHub repo is missing required files: "
        + ", ".join(missing_files)
        + ". Push the latest project changes to GitHub, then rerun this notebook."
    )

print("Project and dependencies are ready.")


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab.output import eval_js

if "your-hf-username/" in HF_REPO_ID:
    raise ValueError("Update HF_REPO_ID in the config cell before launching the app.")

if "app_process" in globals() and app_process.poll() is None:
    app_process.terminate()
    time.sleep(2)

log_path = PROJECT_DIR / "runtime_generated_colab" / "streamlit.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
if log_path.exists():
    log_path.unlink()

env = os.environ.copy()
env.update(
    {
        "EPIGRANET_HF_REPO_ID": HF_REPO_ID,
        "EPIGRANET_HF_REVISION": HF_REVISION,
        "EPIGRANET_HF_MODEL_FILENAME": MODEL_FILENAME,
        "EPIGRANET_HF_EMBEDDINGS_FILENAME": EMBEDDINGS_FILENAME,
        "EPIGRANET_HF_CLASS_MAPPING_FILENAME": CLASS_MAPPING_FILENAME,
        "EPIGRANET_GENERATED_DIR": str(PROJECT_DIR / "runtime_generated_colab"),
    }
)

if HF_TOKEN:
    env["EPIGRANET_HF_TOKEN"] = HF_TOKEN

with open(log_path, "w", encoding="utf-8") as log_file:
    app_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "streamlit_app.py",
            "--server.port",
            str(STREAMLIT_PORT),
            "--server.address",
            "0.0.0.0",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
            "--browser.gatherUsageStats",
            "false",
        ],
        cwd=str(PROJECT_DIR),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

time.sleep(8)
if app_process.poll() is not None:
    print("Streamlit exited during startup. Log output:\n")
    print(log_path.read_text(encoding="utf-8"))
    raise RuntimeError("Streamlit did not stay running. Check the log above.")

streamlit_url = eval_js(f"google.colab.kernel.proxyPort({STREAMLIT_PORT})")
print("Open the Streamlit app here:")
print(streamlit_url)
print(f"Logs: {log_path}")


In [ ]:
log_path = PROJECT_DIR / "runtime_generated_colab" / "streamlit.log"
if log_path.exists():
    print(log_path.read_text(encoding="utf-8"))
else:
    print("No Streamlit log file found yet.")


Use the printed URL to open the app. The Streamlit interface will let you upload an inscription image,
run OCR, inspect preprocessing and segmentation outputs, and download TXT, JSON, or CSV results.


In [ ]:
if "app_process" in globals() and app_process.poll() is None:
    app_process.terminate()
    print("Stopped the Streamlit app.")
else:
    print("No running Streamlit app was found. Run the launch cell first, or open the logs cell to see why startup failed.")
